# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanwajid09/Flyrank-intern/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Unit of Analysis & Time Window
* **Unit of Analysis (Grain):** One row represents a single content page (`content_hash_id`) for a specific client (`client_hash_id`).
* **Time Window:** We use a historical performance window of 30 days. To build features for a target month (e.g. `2026-06`), we look at the last 15 days (recent performance) and compare it against the previous 15 days (past baseline performance).


In [2]:
# No code required here


### 2. Fields Classification (Data Contract)
* **Features (Inputs):**
  1. `imp_last30`: Total GSC impressions in the last 15 days. *Knowable because it is historically logged.*
  2. `clk_last30`: Total GSC clicks in the last 15 days. *Knowable because it is historically logged.*
  3. `pos_last30`: Average GSC position in the last 15 days. *Knowable because it is historically logged.*
  4. `visible_queries`: Number of distinct search queries the page ranks for. *Knowable from the 90-day query dimension table.*
  5. `days_since_last_update`: Days since the content was last modified. *Knowable from content publishing logs.*
* **Label (Target):** `is_declining_label` (1 if organic clicks trend is down, 0 otherwise).
* **Context (Identifiers):** `content_hash_id`, `client_hash_id`.
* **Deliberately Excluded & Rationale:**
  - `trend_pct` and `trend_direction` are excluded from the feature set because they are derived directly from the clicks trend. Using them would cause data leakage.
  - Content pages with `imp_prev30 < 100` are excluded from training because low-volume pages are highly volatile and introduce noise.


In [4]:
# No code required here


## 3. Verify it with queries (grain, counts, missing values, windows)
Below, we run verification queries on the Hugging Face dataset and execute the leakage experiment.


In [6]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# 1. Connect DuckDB to the Hugging Face release
con = duckdb.connect()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# 2. Query 1: Verify the Grain (client_hash_id + content_hash_id is unique)
grain_check = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows, 
        COUNT(DISTINCT content_hash_id) AS unique_content
    FROM {TABLES['dim_content']}
""").df()
print("--- Query 1: Verify Grain ---")
print(grain_check.to_string(index=False))

# 3. Query 2: Row count and date span for mid-panel month using the sample dataset
# We use fact_daily_sample to avoid network timeouts and HF rate limits
month_check = con.sql(f"""
    SELECT 
        COUNT(*) AS row_count, 
        MIN(report_date) AS min_date, 
        MAX(report_date) AS max_date
    FROM {TABLES['fact_daily_sample']}
""").df()
print("\n--- Query 2: Row Count & Date Span (Sample Month: June 2026) ---")
print(month_check.to_string(index=False))

# 4. Query 3: Availability (GA4 Data Available)
availability = con.sql(f"""
    SELECT 
        ga4_data_available, 
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily_sample']}
    GROUP BY ga4_data_available
""").df()
print("\n--- Query 3: GA4 Data Availability (Sample Month: June 2026) ---")
print(availability.to_string(index=False))

# 5. Build Feature Frame using the fast Sample dataset (June 2026)
data_df = con.sql(f"""
    WITH bounds AS (
        SELECT CAST('2026-06-30' AS DATE) AS end_d
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 15 DAY THEN f.gsc_avg_position END)       AS pos_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 15 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30
        FROM {TABLES['fact_daily_sample']} f
        CROSS JOIN bounds b
        WHERE f.report_date > b.end_d - INTERVAL 30 DAY AND f.report_date <= b.end_d
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    ),
    metadata AS (
        SELECT content_hash_id, content_updated_date
        FROM {TABLES['dim_content']}
    ),
    qsignals AS (
        SELECT content_hash_id,
               MAX(content_visible_query_count) AS visible_queries
        FROM {TABLES['fact_query_90d']}
        GROUP BY content_hash_id
    )
    SELECT w.*, 
           COALESCE(date_diff('day', m.content_updated_date, CAST('2026-06-30' AS DATE)), 365) AS days_since_last_update,
           COALESCE(q.visible_queries, 0) AS visible_queries,
           -- Define target: decline in clicks
           CASE WHEN w.clk_last30 < w.clk_prev30 THEN 1 ELSE 0 END AS is_declining_label,
           -- Define leaky column (percentage change in clicks)
           CASE WHEN w.clk_prev30 > 0 THEN (w.clk_last30 - w.clk_prev30) / w.clk_prev30 ELSE 0 END AS leaky_trend_pct
    FROM windowed w
    LEFT JOIN metadata m ON w.content_hash_id = m.content_hash_id
    LEFT JOIN qsignals q ON w.content_hash_id = q.content_hash_id
""").df()

print(f"\n--- Feature Frame Built: {len(data_df):,} rows ---")
print(data_df[["content_hash_id", "imp_last30", "clk_last30", "pos_last30", "days_since_last_update", "visible_queries", "is_declining_label"]].head(3))

# 6. Leakage Experiment
features = ["imp_last30", "clk_last30", "pos_last30", "days_since_last_update", "visible_queries"]
X = data_df[features].fillna(0)
y = data_df["is_declining_label"]

# Train honest tree
clf_honest = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_honest.fit(X, y)
honest_pred = clf_honest.predict(X)
honest_precision = precision_score(y, honest_pred, zero_division=0)
print(f"\nHonest Model Precision: {honest_precision:.4f}")

# Train leaky tree
X_leaky = data_df[features + ["leaky_trend_pct"]].fillna(0)
clf_leaky = DecisionTreeClassifier(max_depth=2, random_state=42)
clf_leaky.fit(X_leaky, y)
leaky_pred = clf_leaky.predict(X_leaky)
leaky_precision = precision_score(y, leaky_pred, zero_division=0)
print(f"Leaky Model Precision: {leaky_precision:.4f}  <- (Score inflated due to target leakage!)")



--- Query 1: Verify Grain ---
 total_rows  unique_content
     519606          519606
--- Query 2: Row Count & Date Span (Sample Month: June 2026) ---
 row_count   min_date   max_date
  11694072 2026-06-01 2026-06-30
--- Query 3: GA4 Data Availability (Sample Month: June 2026) ---
 ga4_data_available  row_count
              False    8651918
               True     644726
               <NA>    2397428
--- Feature Frame Built: 79,564 rows ---
            content_hash_id  imp_last30  ...  visible_queries  is_declining_label
0  content_bff3dd7b6cef629c       472.0  ...               15                   0
1  content_1bc67e9d435b3a78       232.0  ...                3                   0
2  content_a28fcf5ddf094229       126.0  ...                6                   0
[3 rows x 7 columns]
Honest Model Precision: 0.5734
Leaky Model Precision: 1.0000  <- (Score inflated due to target leakage!)


### 4. Data Limitations
* **Unbalanced history:** Different clients have different tracking start dates. This means windowing must be handled carefully per client rather than globally.
* **GA4 Missingness:** Early history or specific clients lack GA4 events (`ga4_data_available` is false), meaning we can only rely on GSC search metrics for those segments.
* **Low Volume Noise:** Pages with very low search volume have highly volatile CTR and position shifts, which can easily trick the model if they are not filtered out.


In [8]:
# No code required here


## Self-check
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
